In [1]:
print("OK")

OK


In [1]:
from langchain_core.prompts import PromptTemplate

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

In [11]:
from langchain.chains import create_retrieval_chain

In [2]:
import ssl

# 1. Store the original method that loads system certificates
orig_load_default_certs = ssl.SSLContext.load_default_certs

# 2. Define a dummy method that does nothing to bypass the corruption
def dummy_load_default_certs(self, purpose=ssl.Purpose.SERVER_AUTH):
    pass

# 3. Patch the SSLContext with our safe dummy method
ssl.SSLContext.load_default_certs = dummy_load_default_certs

# ==========================================
# NOW RUN YOUR IMPORTS AND RUNNABLES SAFELY
# ==========================================
import os
import certifi

# Force network requests to use a clean certifi bundle instead
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

# This import will now bypass the broken Windows store and work!
from langchain_pinecone import PineconeVectorStore

c:\Users\shrey\anaconda3\envs\mchatbot\lib\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [21]:
from langchain_pinecone import PineconeVectorStore

In [22]:
import pinecone

In [28]:
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [26]:
from langchain_community.llms import CTransformers

In [31]:
from dotenv import load_dotenv
load_dotenv()

#access the environment variable
KEY=os.getenv("PINECONE_API_KEY")

In [33]:
def load_pdf(data):
    loader = DirectoryLoader(data, glob="**/*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [35]:
extracted_data=load_pdf("C:/Users/shrey/OneDrive/Desktop/Medical-Chatbot-using-Llama2/data/")

In [36]:
#text chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [37]:
text_chunks=text_split(extracted_data)
print("Length of chunks:",len(text_chunks))

Length of chunks: 40000


In [38]:
#download embeddings model
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [39]:
embeddings=download_hugging_face_embeddings()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
c:\Users\shrey\anaconda3\envs\mchatbot\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shrey\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to a

In [40]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [41]:
query_result=embeddings.embed_query("Hello World")
print("length:",len(query_result))

length: 384


In [43]:
from pinecone import Pinecone
pc=Pinecone(api_key=KEY)
index_name="testing-pinecone"

In [45]:
#creating embeddings for each text chunks & store
docsearch=PineconeVectorStore.from_texts(
    [t.page_content for t in text_chunks],
    embeddings,
    index_name=index_name,
    pinecone_api_key=KEY
)

ProtocolError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [46]:
docsearch=PineconeVectorStore.from_existing_index(index_name,embeddings)
query="What are allergies?"
docs=docsearch.similarity_search(query,k=3)
print("Result:",docs)

Result: [Document(page_content='‘‘What’s New in: Asthma and Allergic Rhinitis.’’Pulse\n(September 20, 2004): 50.\nRichard Robinson\nTeresa G. Odle\nAllergies\nDefinition\nAllergies are abnormal reactions of the immune\nsystem that occur in response to otherwise harmless\nsubstances.\nDescription\nAllergies are among the most common of medical\ndisorders. It is estimated that 60 million Americans, or\nmore than one in every five people, suffer from some\nform of allergy, with similar proportions throughout'), Document(page_content='triggered by harmless, everyday substances. This is\nthe condition known as allergy, and the offending\nsubstance is called an allergen. Common inhaled\nallergens include pollen,dust, and insect parts from\ntiny house mites. Common food allergens include\nnuts, fish, and milk.\nAllergic reactions involve a special set of cells\nin the immune system known as mast cells. Mast\ncells serve as guards in the tissues where the body\nmeets the outside world: the ski

In [58]:
prompt_template="""
Use the following pieces of information to answer the user's question.If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:{context}
Question:{input}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

In [59]:
PROMPT=PromptTemplate(
    template=prompt_template,
    input_variables=["context","input"]
)

In [60]:
llm=CTransformers(model="model/llama-2-7b-chat.ggmlv3.q4_0.bin",model_type="llama",config={'max_new_tokens':512,'temperature':0.8})

In [61]:
from langchain.chains.combine_documents import create_stuff_documents_chain

In [62]:
question_answer_chain = create_stuff_documents_chain(llm, PROMPT)

In [63]:
retriever = docsearch.as_retriever(search_kwargs={'k': 2})
qa = create_retrieval_chain(retriever, question_answer_chain)

In [64]:
while True:
    user_input=input(f"Input Prompt:")
    if user_input.lower() in ['exit', 'quit', 'q']:
        print("Exiting chatbot. Goodbye!")
        break
        
    if not user_input.strip():
        continue

    result = qa.invoke({"input": user_input})
    
    print("Response:", result["answer"])

Response: Acne is a skin disorder that occurs when the oil-producing glands within the skin, known as sebaceous glands, become inflamed. This can lead to the formation of pimples, which are small bumps on the skin that can be red, white or black in color. Acne can occur on the face, chest, back and other areas of the body. It is a common condition that affects many people, especially during adolescence and young adulthood.
Exiting chatbot. Goodbye!
